In [2]:
import numpy as np
import math
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter, defaultdict
from nltk.translate.bleu_score import sentence_bleu
from nltk.util import ngrams

print("Dependencies loaded.")

# --- SHARED DATASET ---
# A small dummy corpus for demonstration
corpus = [
    "the cat sat on the mat",
    "the dog ate the bone",
    "the cat ate the mouse",
    "the dog sat on the log"
]

# Preprocessing: Tokenization
tokenized_corpus = [sentence.split() for sentence in corpus]
vocab = set(word for sentence in tokenized_corpus for word in sentence)
vocab.add("<UNK>")
vocab.add("<EOS>")
word_to_idx = {word: i for i, word in enumerate(sorted(list(vocab)))}
idx_to_word = {i: word for word, i in word_to_idx.items()}
vocab_size = len(vocab)

print(f"Vocabulary: {vocab}")

# --- PART 1: N-GRAM MODEL (Bigram) ---
print("\n" + "="*40)
print("PART 1: N-Gram Model (Bigram)")
print("="*40)

class BigramModel:
    def __init__(self):
        self.bigram_counts = defaultdict(Counter)
        self.unigram_counts = Counter()

    def train(self, corpus):
        for sentence in corpus:
            # Add start token implicitly or just use raw pairs
            # Here we just use raw pairs for simplicity
            for i in range(len(sentence) - 1):
                w1, w2 = sentence[i], sentence[i+1]
                self.bigram_counts[w1][w2] += 1
                self.unigram_counts[w1] += 1

    def calculate_perplexity(self, sentence):
        N = len(sentence)
        log_prob_sum = 0
        for i in range(N - 1):
            w1, w2 = sentence[i], sentence[i+1]
            # Laplace smoothing (add-1)
            count_w1_w2 = self.bigram_counts[w1][w2] + 1
            count_w1 = self.unigram_counts[w1] + vocab_size
            prob = count_w1_w2 / count_w1
            log_prob_sum += math.log(prob)

        # Perplexity = exp(-1/N * sum(log_probs))
        return math.exp(-log_prob_sum / (N-1))

# Train Bigram Model
ngram_model = BigramModel()
ngram_model.train(tokenized_corpus)

# Test N-gram Metrics
test_sentence = "the cat sat on the log".split()
reference = [["the", "cat", "sat", "on", "the", "mat"], ["the", "dog", "sat", "on", "the", "log"]]

# 1. Perplexity
ngram_ppl = ngram_model.calculate_perplexity(test_sentence)
# 2. BLEU Score (comparing test sentence to reference sentences)
ngram_bleu = sentence_bleu(reference, test_sentence)

print(f"Test Sentence: {test_sentence}")
print(f"N-Gram Perplexity: {ngram_ppl:.4f}")
print(f"N-Gram BLEU Score: {ngram_bleu:.4f}")


# --- PART 2: RNN LANGUAGE MODEL (PyTorch) ---
print("\n" + "="*40)
print("PART 2: RNN Language Model")
print("="*40)

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(SimpleRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x)
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

# Hyperparameters
embed_size = 16
hidden_size = 32
model = SimpleRNN(vocab_size, embed_size, hidden_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Prepare Data for Training (Input: "the cat", Target: "cat sat")
inputs = []
targets = []
for sentence in tokenized_corpus:
    # Convert words to indices
    indices = [word_to_idx[w] for w in sentence]
    for i in range(len(indices) - 1):
        inputs.append(indices[i])
        targets.append(indices[i+1])

inputs = torch.tensor(inputs).unsqueeze(1) # Shape: [Total_Words, 1]
targets = torch.tensor(targets)            # Shape: [Total_Words]

# Train RNN (Simple loop)
print("Training RNN...")
for epoch in range(50):
    hidden = None
    optimizer.zero_grad()
    output, hidden = model(inputs, hidden)

    # Reshape output for Loss
    loss = criterion(output.view(-1, vocab_size), targets)
    loss.backward()
    optimizer.step()

print("Training complete.")

# --- RNN Metrics ---

def calculate_rnn_perplexity(model, sentence):
    model.eval()
    idx_sentence = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in sentence]
    input_tensor = torch.tensor(idx_sentence[:-1]).unsqueeze(1)
    target_tensor = torch.tensor(idx_sentence[1:])

    hidden = None
    with torch.no_grad():
        output, _ = model(input_tensor, hidden)
        loss = criterion(output.view(-1, vocab_size), target_tensor)

    # Perplexity is exp(CrossEntropyLoss)
    return torch.exp(loss).item()

# 1. Perplexity
rnn_ppl = calculate_rnn_perplexity(model, test_sentence)

# 2. BLEU Score
# For a Language Model, BLEU usually compares generated text to a reference.
# Let's generate a short sequence starting with "the"
start_word = "the"
generated_sentence = [start_word]
input_idx = torch.tensor([[word_to_idx[start_word]]])
hidden = None

for _ in range(5): # Generate 5 words
    with torch.no_grad():
        output, hidden = model(input_idx, hidden)
        predicted_idx = output.argmax(dim=2).item()
        word = idx_to_word[predicted_idx]
        generated_sentence.append(word)
        input_idx = torch.tensor([[predicted_idx]])

rnn_bleu = sentence_bleu(reference, generated_sentence)

print(f"Test Sentence (for PPL): {test_sentence}")
print(f"Generated Sentence (for BLEU): {generated_sentence}")
print(f"RNN Perplexity: {rnn_ppl:.4f}")
print(f"RNN BLEU Score: {rnn_bleu:.4f}")

Dependencies loaded.
Vocabulary: {'on', '<UNK>', 'sat', 'ate', 'mat', 'log', 'mouse', 'bone', 'cat', '<EOS>', 'dog', 'the'}

PART 1: N-Gram Model (Bigram)
Test Sentence: ['the', 'cat', 'sat', 'on', 'the', 'log']
N-Gram Perplexity: 6.3300
N-Gram BLEU Score: 1.0000

PART 2: RNN Language Model
Training RNN...
Training complete.
Test Sentence (for PPL): ['the', 'cat', 'sat', 'on', 'the', 'log']
Generated Sentence (for BLEU): ['the', 'dog', 'sat', 'on', 'the', 'dog']
RNN Perplexity: 2.3232
RNN BLEU Score: 0.7598
